# Multi-Agent VLM Evaluation

Orchestrate an autonomous Multi-Agent system to solve complex reasoning workflows while integrating tool use and an evaluation layer for a Vision-Language Model.

In [1]:
import json
import base64
import requests
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage

## Tool Setup

In [2]:
def calculate_stress_tool(weight_kg, area_m2):
    try:
        stress = float(weight_kg) / float(area_m2)
        return f"Calculated structural stress: {stress} Pascals"
    except Exception as e:
        return f"Calculation error: {e}"

tool_registry = {
    "calculate_stress": calculate_stress_tool
}

## Agent Definitions

In [3]:
def reasoning_agent(query):
    llm = ChatOllama(model="lfm3:1.7b", temperature=0)
    prompt = (
        f"You have access to a tool called 'calculate_stress(weight_kg, area_m2)'. "
        f"Based on the query: '{query}', extract the numbers and output ONLY a valid JSON object "
        f"with keys 'weight_kg' and 'area_m2'. Do not add any extra text."
    )
    response = llm.invoke(prompt).content.strip()
    
    try:
        parsed_args = json.loads(response)
        return tool_registry["calculate_stress"](**parsed_args)
    except:
        return "Failed to parse tool arguments."

def vision_agent(image_url, context):
    vlm = ChatOllama(model="llava", temperature=0)
    
    encoded_image = base64.b64encode(requests.get(image_url).content).decode('utf-8')
    
    message = HumanMessage(
        content=[
            {"type": "text", "text": f"Context: {context}. Analyze this image and determine if the structure can support this stress."},
            {"type": "image_url", "image_url": f"data:image/jpeg;base64,{encoded_image}"},
        ]
    )
    return vlm.invoke([message]).content

def evaluation_agent(reasoning_output, vision_output):
    llm = ChatOllama(model="lfm3:1.7b", temperature=0)
    prompt = (
        f"Evaluate the system's reasoning.\n"
        f"Step 1 (Tool Output): {reasoning_output}\n"
        f"Step 2 (VLM Output): {vision_output}\n"
        f"Provide a concise evaluation and a final score out of 10 for the accuracy and logical flow."
    )
    return llm.invoke(prompt).content

## Orchestration Engine

In [4]:
class AutonomousOrchestrator:
    def __init__(self, query, image_url):
        self.query = query
        self.image_url = image_url
        self.state = {}

    def execute_workflow(self):
        self.state["reasoning_result"] = reasoning_agent(self.query)
        
        self.state["vision_result"] = vision_agent(
            self.image_url, 
            self.state["reasoning_result"]
        )
        
        self.state["evaluation_result"] = evaluation_agent(
            self.state["reasoning_result"], 
            self.state["vision_result"]
        )
        
        return self.state

## Execution

In [5]:
task_query = "The steel beam holds a weight of 5000 kg over an area of 2.5 square meters."
test_image_url = "https://images.unsplash.com/photo-1518458028785-8fbcd101ebb9?w=500&q=80"

orchestrator = AutonomousOrchestrator(query=task_query, image_url=test_image_url)
results = orchestrator.execute_workflow()

print("=== Stage 1: Reasoning & Tool Use ===")
print(results["reasoning_result"])

print("\n=== Stage 2: Vision-Language Analysis ===")
print(results["vision_result"])

print("\n=== Stage 3: Autonomous Evaluation ===")
print(results["evaluation_result"])

=== Stage 1: Reasoning & Tool Use ===
Calculated structural stress: 2000.0 Pascals

=== Stage 2: Vision-Language Analysis ===
 The image shows a person holding a wad of money, which appears to be a significant amount. The structural stress mentioned in the context is 2000.0 Pascals, which is a very low value, indicating that the structure is not under any significant load.

In the context of structural stress, the image does not provide any information about the structural integrity of the building or the materials it is made of. Structural stress is typically measured in units of force per unit area, such as Pascals (Pa) or pounds per square inch (psi).

To determine if the structure can support this stress, we would need to know the material properties of the structure, the dimensions of the structure, and the method of loading (how the weight is distributed). Without this information, it is not possible to make a definitive assessment of the structure's ability to support the stress